# 02: Data Pre-Processing & Feature Engineering

In [2]:
import os
import numpy as np
import pandas as pd

In [3]:
HERE = os.getcwd()
DATA_DIR = os.path.join(HERE, "data")
IN_PATH = os.path.join(DATA_DIR, "transaction_dataset.csv")
OUT_PATH = os.path.join(DATA_DIR, "pre_process_transaction_ds.csv")

In [4]:
df = pd.read_csv(IN_PATH)
df.head()

,Unnamed: 0,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


In [8]:
FEATURE_NAMES = [
    "txVolume",            # tanh(log1p(avg value moved in ETH) / 3)
    "txFrequency",         # tanh(tx count / 50)
    "accountAge",          # tanh(active days / 365)
    "networkDegree",       # tanh(unique counterparties / 20)
    "timeRegularity",      # tanh(log1p(avg minutes between txs) / 6)
    "valueSentRatio",      # sent value / (sent + received)  -> 0..1, 0.5 neutral
    "inOutRatio",          # received txns / total txns      -> 0..1
    "degreeConcentration", # unique counterparties / tx count -> 0..1
    "valueVolatility",     # tanh(log1p(max value moved) / 3)
]

In [5]:
X = pd.DataFrame(index=df.index)

tx_count = pd.to_numeric(df["total transactions (including tnx to create contract"], errors="coerce")
sent_value = pd.to_numeric(df["total Ether sent"], errors="coerce")
received_value = pd.to_numeric(df["total ether received"], errors="coerce")
received_count = pd.to_numeric(df["Received Tnx"], errors="coerce")
unique_sent = pd.to_numeric(df["Unique Sent To Addresses"], errors="coerce")
unique_received = pd.to_numeric(df["Unique Received From Addresses"], errors="coerce")
max_sent = pd.to_numeric(df["max val sent"], errors="coerce")
max_received = pd.to_numeric(df["max value received "], errors="coerce")

total_value = sent_value + received_value
avg_value = np.where(tx_count > 0, total_value / tx_count.clip(lower=1), 0.0)
age_days = pd.to_numeric(df["Time Diff between first and last (Mins)"], errors="coerce") / 1440.0
gap_minutes = pd.to_numeric(df["Avg min between sent tnx"], errors="coerce")
unique_total = unique_sent + unique_received

X["txVolume"] = np.tanh(np.log1p(avg_value) / 3.0)
X["txFrequency"] = np.tanh(tx_count / 50.0)
X["accountAge"] = np.tanh(age_days / 365.0)
X["networkDegree"] = np.tanh(unique_total / 20.0)
X["timeRegularity"] = np.tanh(np.log1p(gap_minutes.fillna(0.0)) / 6.0)
sent_ratio_denom = sent_value + received_value
X["valueSentRatio"] = np.where(sent_ratio_denom > 0, sent_value / sent_ratio_denom.replace(0, np.nan), 0.5)
X["inOutRatio"] = np.where(tx_count > 0, received_count / tx_count.clip(lower=1), 0.5)
X["degreeConcentration"] = np.where(tx_count > 0, unique_total / tx_count.clip(lower=1), 0.0)
X["valueVolatility"] = np.tanh(np.log1p(max_sent.fillna(0.0) + max_received.fillna(0.0)) / 3.0)

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

In [6]:
X["FLAG"] = df["FLAG"].to_numpy()

keep = X.drop(columns=["FLAG"]).notna().all(axis=1)
print(f"Rows with complete features: {int(keep.sum()):,} / {len(df):,} (dropped {int((~keep).sum()):,})")
print(f"Label balance: fraud={int((X['FLAG'][keep] == 1).sum()):,} legit={int((X['FLAG'][keep] == 0).sum()):,}")

pre = X[keep].reset_index(drop=True)
pre.to_csv(OUT_PATH, index=False)
print(f"Exported {OUT_PATH} ({os.path.getsize(OUT_PATH):,} bytes, {len(pre):,} rows)")

Rows with complete features: 9,841 / 9,841 (dropped 0)
Label balance: fraud=2,179 legit=7,662
Exported /home/ayush/Desktop/code/SecureTransac/ai/data/pre_process_transaction_ds.csv (1,437,570 bytes, 9,841 rows)


In [9]:
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
pre[FEATURE_NAMES].describe().T

,count,mean,std,min,25%,50%,75%,max
txVolume,9841.000,0.412,0.330,0.000,0.113,0.327,0.768,0.996
txFrequency,9841.000,0.378,0.380,0.000,0.080,0.159,0.793,1.000
accountAge,9841.000,0.283,0.338,0.000,0.001,0.088,0.522,0.999
networkDegree,9841.000,0.331,0.288,0.000,0.149,0.197,0.422,1.000
timeRegularity,9841.000,0.418,0.368,0.000,0.000,0.450,0.784,0.974
valueSentRatio,9841.000,0.432,0.184,0.000,0.499,0.500,0.500,1.000
inOutRatio,9841.000,0.555,0.243,0.000,0.400,0.500,0.714,1.000
degreeConcentration,9841.000,0.590,0.392,0.000,0.167,0.667,1.000,1.000
valueVolatility,9841.000,0.606,0.338,0.000,0.315,0.706,0.932,1.000


In [10]:
# Separation check: do features actually differ between fraud and legit?
pre.groupby("FLAG")[FEATURE_NAMES].mean().T.round(3).rename(
    columns={0: "legit mean", 1: "fraud mean"}
)

FLAG,legit mean,fraud mean
txVolume,0.475,0.188
txFrequency,0.426,0.209
accountAge,0.336,0.095
networkDegree,0.333,0.322
timeRegularity,0.459,0.272
valueSentRatio,0.424,0.459
inOutRatio,0.527,0.651
degreeConcentration,0.576,0.638
valueVolatility,0.671,0.376
